In [4]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from data_io import DataIO #Custom IO file
run_label = '2024-07-15' ## change this as needed
path = "/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20240715/OutgassingData_{}.h5".format(run_label, run_label)
IO = DataIO(path)

In [5]:
print(path)

/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20240715/OutgassingData_2024-07-15.h5


In [6]:
IO.GetTimeData()
timedataframe = IO.timedata
print(timedataframe)

0   2024-07-15 14:13:27
0   2024-07-15 14:13:57
0   2024-07-15 14:14:28
0   2024-07-15 14:14:59
0   2024-07-15 14:15:30
            ...        
0   2024-07-19 02:10:34
0   2024-07-19 02:11:04
0   2024-07-19 02:11:35
0   2024-07-19 02:12:06
0   2024-07-19 02:12:37
Length: 9844, dtype: datetime64[ns]


In [7]:
IO.GetTECData()
tecdataframe = IO.tecdata
print(tecdataframe)

    Set Temp
0        0.0
0        0.0
0        0.0
0        0.0
0        0.0
..       ...
0       50.0
0       50.0
0       50.0
0       50.0
0       50.0

[9844 rows x 1 columns]


In [8]:
IO.GetPressureData()
pressuredataframe = IO.pressuredata
print(pressuredataframe)

    Total Pressure
0     9.490000e-05
0     9.620000e-05
0     9.710000e-05
0     9.620000e-05
0     9.490000e-05
..             ...
0     7.880000e-07
0     7.880000e-07
0     7.880000e-07
0     7.880000e-07
0     7.910000e-07

[9844 rows x 1 columns]


In [9]:
IO.GetOmegaData()
omegadataframe = IO.omegadata
print(omegadataframe)

     CH1   CH2
0   23.8  23.7
0   23.8  23.7
0   23.8  23.6
0   23.9  23.7
0   23.9  23.7
..   ...   ...
0   47.7  46.3
0   47.7  46.3
0   47.7  46.3
0   47.7  46.3
0   47.6  46.3

[9844 rows x 2 columns]


In [10]:
IO.GetRGAData()
rgadataframe = IO.rgadata
rgadataframe[rgadataframe < 0] = 1e-15 #negative pressure values recorded due to noise fluctutation below the RGA's sensitivity
print(rgadataframe['32.00amu'])

data run = 1       4.794190e-08
data run = 2       4.699535e-08
data run = 3       4.349200e-08
data run = 4       4.358035e-08
data run = 5       4.384880e-08
                       ...     
data run = 9840    1.670950e-09
data run = 9841    1.707050e-09
data run = 9842    1.728900e-09
data run = 9843    1.654550e-09
data run = 9844    1.636400e-09
Name: 32.00amu, Length: 9844, dtype: float64


# Data manipulation

In [11]:
start_datetime = timedataframe.iloc[0]
gases = ['H2', 'H2O', 'N2', 'O2', 'CO2']
gas_masses = ['2.00amu', '18.00amu', '28.00amu', '32.00amu', '44.00amu']

In [12]:
final_tables = []
for idx, gas_mass in enumerate(gas_masses):
    time_column = (timedataframe - start_datetime) / np.timedelta64(1, 's')
    rga_column = rgadataframe[gas_masses[idx]].reset_index(drop=True)  # Reset index of rga_column to prevent NaN
    pressure_column = pressuredataframe
    temp1_column = omegadataframe['CH1']
    temp2_column = omegadataframe['CH2']
    tec_column = tecdataframe
    
    # Collecting columns in a dataframe
    gas_final_data = pd.DataFrame(data=time_column)
    gas_final_data.columns = ['Exposure_time']
    
    # Merge rga_column with gas_final_data and fill NaN values
    gas_final_data['Partial_pressure'] = rga_column.values #the attribute .values prevents the index resetting from duplicate values
    
    gas_final_data['Total_pressure'] = pressure_column
    
    # Celsius to Kelvin
    gas_final_data['CH1_temp'] = temp1_column + 273.2
    gas_final_data['CH2_temp'] = temp2_column + 273.2
    gas_final_data['Mean_temp'] = (gas_final_data['CH1_temp'] + gas_final_data['CH2_temp']) / 2.0
    gas_final_data['TEC_temp'] = tec_column
    
    final_tables.append(gas_final_data)
    print(gas_final_data)

    Exposure_time  Partial_pressure  Total_pressure  CH1_temp  CH2_temp  \
0             0.0      3.978760e-08    9.490000e-05     297.0     296.9   
0            30.0      3.709320e-08    9.620000e-05     297.0     296.9   
0            61.0      3.236330e-08    9.710000e-05     297.0     296.8   
0            92.0      3.194515e-08    9.620000e-05     297.1     296.9   
0           123.0      3.079210e-08    9.490000e-05     297.1     296.9   
..            ...               ...             ...       ...       ...   
0        302227.0      9.280000e-11    7.880000e-07     320.9     319.5   
0        302257.0      9.845000e-11    7.880000e-07     320.9     319.5   
0        302288.0      1.446000e-10    7.880000e-07     320.9     319.5   
0        302319.0      9.750000e-11    7.880000e-07     320.9     319.5   
0        302350.0      9.835000e-11    7.910000e-07     320.8     319.5   

    Mean_temp  TEC_temp  
0      296.95       0.0  
0      296.95       0.0  
0      296.90       0

In [13]:
new_path = "/gpfs/gibbs/project/david_moore/aj487/Data_WL110/Outgassing_Setup/20240715"
hdf_name = '{}/{}.h5'.format(new_path, run_label)
for idx, gas in enumerate(gases):
    final_tables[idx].sort_values(by='Exposure_time', inplace=True) # one more sort just to be sure
    final_tables[idx].to_hdf(hdf_name, key=gas)
print(final_tables)

[    Exposure_time  Partial_pressure  Total_pressure  CH1_temp  CH2_temp  \
0             0.0      3.978760e-08    9.490000e-05     297.0     296.9   
0            30.0      3.709320e-08    9.620000e-05     297.0     296.9   
0            61.0      3.236330e-08    9.710000e-05     297.0     296.8   
0            92.0      3.194515e-08    9.620000e-05     297.1     296.9   
0           123.0      3.079210e-08    9.490000e-05     297.1     296.9   
..            ...               ...             ...       ...       ...   
0        302227.0      9.280000e-11    7.880000e-07     320.9     319.5   
0        302257.0      9.845000e-11    7.880000e-07     320.9     319.5   
0        302288.0      1.446000e-10    7.880000e-07     320.9     319.5   
0        302319.0      9.750000e-11    7.880000e-07     320.9     319.5   
0        302350.0      9.835000e-11    7.910000e-07     320.8     319.5   

    Mean_temp  TEC_temp  
0      296.95       0.0  
0      296.95       0.0  
0      296.90       